# Notebook 5 — Feature Engineering
Create production-safe features from the EDA decisions. Transformations are fitted on training data and then applied to validation/test. Fitted objects are saved so production can reuse the exact same transformations.

In [ ]:

import os, warnings
from pathlib import Path
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

ART = Path("../artifacts")
ART.mkdir(exist_ok=True)
CHARTS = ART / "charts"
CHARTS.mkdir(exist_ok=True)

import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.read_csv(ART/"03_train.csv")
val = pd.read_csv(ART/"03_validation.csv")
test = pd.read_csv(ART/"03_test.csv")

for d in [train,val,test]:
    d["order_purchase_timestamp"] = pd.to_datetime(d["order_purchase_timestamp"], errors="coerce")


In [ ]:

def make_features(df):
    x = pd.DataFrame(index=df.index)
    dt = df["order_purchase_timestamp"]
    x["purchase_year"] = dt.dt.year
    x["purchase_month"] = dt.dt.month
    x["purchase_dayofweek"] = dt.dt.dayofweek
    x["purchase_hour"] = dt.dt.hour

    for c in ["order_item_count","total_freight_value","unique_products","unique_sellers",
              "payment_count","total_payment_value","payment_installments","seller_count","seller_states"]:
        if c in df.columns:
            x[c] = pd.to_numeric(df[c], errors="coerce")

    for c in ["customer_state","seller_state","order_status"]:
        if c in df.columns:
            x[c] = df[c].astype("object")
    return x

target = "late"
X_train = make_features(train)
X_val = make_features(val)
X_test = make_features(test)
y_train, y_val, y_test = train[target], val[target], test[target]

print("Features:", list(X_train.columns))


In [ ]:

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

Xtr = preprocessor.fit_transform(X_train)
Xv = preprocessor.transform(X_val)
Xte = preprocessor.transform(X_test)

joblib.dump(preprocessor, ART/"05_preprocessor.joblib")
pd.DataFrame({"feature": X_train.columns}).to_csv(ART/"05_feature_list.csv", index=False)

print("Transformed shapes:", Xtr.shape, Xv.shape, Xte.shape)


In [ ]:

# Save transformed arrays for reproducibility.
from scipy import sparse
sparse.save_npz(ART/"05_X_train.npz", Xtr)
sparse.save_npz(ART/"05_X_validation.npz", Xv)
sparse.save_npz(ART/"05_X_test.npz", Xte)
np.save(ART/"05_y_train.npy", y_train.to_numpy())
np.save(ART/"05_y_validation.npy", y_val.to_numpy())
np.save(ART/"05_y_test.npy", y_test.to_numpy())
print("Saved fitted transformer, feature list, and transformed data artifacts.")
